# Iterative Workflow: Tweet Writer that Improves Itself

One LLM writes a tweet, another judges it. If it's not approved, the feedback goes to an
optimizer that rewrites it, and the judge checks again. Stops when approved or after `max_iteration` rounds.

```
START -> generate -> evaluate --(approved / max reached)--> END
                       ^   --(needs_improvement)--> optimize
                       |                               |
                       +-------------------------------+
```

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import operator

load_dotenv()

generator_model = ChatGroq(model='openai/gpt-oss-20b', reasoning_effort='low')  # low = less hidden thinking, avoids empty replies
evaluator_model = ChatGroq(model='openai/gpt-oss-120b')  # bigger model as the strict judge

In [2]:
class TweetEvaluation(BaseModel):
    evaluation: Literal['approved', 'needs_improvement'] = Field(description='Final verdict')
    feedback: str = Field(description='Short feedback on what to improve')


structured_evaluator = evaluator_model.with_structured_output(TweetEvaluation)

In [3]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: str
    feedback: str
    iteration: int
    max_iteration: int
    tweet_history: Annotated[list[str], operator.add]     # every version, appended each round
    feedback_history: Annotated[list[str], operator.add]

In [4]:
def generate_tweet(state: TweetState):
    prompt = (f"Write a short, funny, original tweet on the topic: {state['topic']}.\n"
              "Max 280 characters. No hashtags. Reply with only the tweet.")
    tweet = generator_model.invoke(prompt).content
    return {'tweet': tweet, 'tweet_history': [tweet]}


def evaluate_tweet(state: TweetState):
    prompt = ("You are a very strict Twitter critic. Judge this tweet on humour, originality "
              "and virality. Approve it only if it is genuinely great.\n"
              f"Tweet: {state['tweet']}")
    result = structured_evaluator.invoke(prompt)
    return {'evaluation': result.evaluation, 'feedback': result.feedback,
            'feedback_history': [result.feedback]}


def optimize_tweet(state: TweetState):
    prompt = (f"Improve this tweet on '{state['topic']}' using the feedback.\n"
              f"Tweet: {state['tweet']}\nFeedback: {state['feedback']}\n"
              "Max 280 characters. No hashtags. Reply with only the new tweet.")
    tweet = generator_model.invoke(prompt).content
    return {'tweet': tweet, 'tweet_history': [tweet], 'iteration': state['iteration'] + 1}

In [5]:
# Two ways out of the loop: approved, or out of attempts
def route_evaluation(state: TweetState) -> Literal['optimize', '__end__']:
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return END
    return 'optimize'

In [6]:
graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', route_evaluation)
graph.add_edge('optimize', 'evaluate')  # loop back to the judge

workflow = graph.compile()

In [7]:
result = workflow.invoke({'topic': 'Indian Railways', 'iteration': 1, 'max_iteration': 4})

for i, (tweet, feedback) in enumerate(zip(result['tweet_history'], result['feedback_history']), 1):
    print(f'--- Version {i} ---')
    print(tweet)
    print('Feedback:', feedback, '\n')

print('Final verdict:', result['evaluation'], 'after', result['iteration'], 'round(s)')

--- Version 1 ---
Just saw a train on Indian Railways with a “Late” sign that reads “Coming Soon.” If the train’s punctual, it’ll be a paradox; if it’s delayed, it’ll be a miracle. Either way, I’m still on the platform, eating samosas, and writing this tweet.
Feedback: The premise is mildly funny and relatable, but the joke feels stretched and the phrasing is clunky. Sharper punchline and tighter wording would boost humor, originality, and shareability. 

--- Version 2 ---
Saw a “Late” sign on an Indian Railway train that said “Coming Soon.” If it’s on time, it’s a paradox; if it’s delayed, a miracle. Either way, I’m still on the platform, munching samosas and tweeting.
Feedback: The tweet is witty and relatable, but the humor leans on a familiar train‑delay trope and the punchline feels a bit flat. To be genuinely great, tighten the paradox line for a sharper twist and add a more unexpected visual or wordplay that will boost shareability. 

--- Version 3 ---
Saw a “Late” sign on an In